# PI3: calidad del modelo cuantizado (8 y 4 bits) — Google Colab

Mide cuánta calidad (BLEU/chrF sobre el test común de 174 entradas) pierde el modelo fusionado al cuantizarlo con bitsandbytes, **en la misma sesión** que la referencia fp16, para que la comparación sea limpia. Ver `evaluation/pi3_cuantizacion_gpu.py`.

Runtime: **GPU T4**. Duración estimada: ~6 min por variante (3 variantes) + instalación. **Esto no mide la latencia en CPU** (eso es `evaluation/pi3_cuantizacion.py`).

In [ ]:
import torch
!nvidia-smi
assert torch.cuda.is_available(), "SIN GPU: cambia el tipo de entorno a T4 GPU"

In [ ]:
import os, json, time
%cd /content
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone -b develop https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm
!git pull -q origin develop
!pip install -q transformers peft accelerate sacrebleu python-dotenv bitsandbytes
!pip uninstall -y -q torchao
os.makedirs("merging/logs", exist_ok=True)
TIEMPOS_PATH = "merging/logs/tiempos_cuantizacion.json"
def registrar(nombre, t0):
    t = json.load(open(TIEMPOS_PATH)) if os.path.exists(TIEMPOS_PATH) else {}
    t[nombre] = round(time.time() - t0, 1)
    json.dump(t, open(TIEMPOS_PATH, "w"), indent=1)
    print(f"[tiempo] {nombre}: {t[nombre] / 60:.1f} min")

## Paso 1 — Regenerar el adaptador del promedio exacto (peft_cat_norm)

El archivo de pesos no está en git, solo la configuración; se reconstruye en ~15 s con los tres adaptadores individuales.

In [ ]:
G = "finetuning/checkpoints/generador{}/adapter"
ADAPTADORES = f"{G.format(1)} {G.format(2)} {G.format(3)}"
t0 = time.time()
!python merging/fusionar_adaptadores.py --metodo cat --normalizar --adaptadores {ADAPTADORES} --salida finetuning/checkpoints/peft_cat_norm 2>&1 | tail -2
registrar("fusion_peft_cat_norm", t0)
assert os.path.exists("finetuning/checkpoints/peft_cat_norm/fusion/adapter_model.safetensors"), "no se generó el adaptador"

## Paso 2 — Traducir el test común con cada variante (fp16, int8, nf4)

Cada celda es reanudable. **Anota la línea JSON que imprime cada una** (memoria de GPU y latencia mediana): sirve para el informe.

In [ ]:
for v in ["fp16", "int8", "nf4"]:
    t0 = time.time()
    !python evaluation/pi3_cuantizacion_gpu.py --variante {v} 2>&1 | tee merging/logs/cuant_{v}.log | tail -3
    registrar(f"cuantizacion_{v}", t0)
    assert os.path.exists(f"evaluation/predicciones_cuant/gpu_{v}.json"), f"{v}: no se generaron predicciones"

## Paso 3 — Informe de calidad (BLEU/chrF con intervalo por semilla)

In [ ]:
!python evaluation/pi3_cuantizacion_gpu.py --informe

## Paso 4 — Empaquetar (solo resultados chicos) y descargar

Se lleva `resultados_cuantizacion.zip` al repo: descomprímelo en una carpeta aparte y avísame; yo reviso, integro y documento.

In [ ]:
!zip -q -r resultados_cuantizacion.zip evaluation/predicciones_cuant/gpu_*.json evaluation/pi3_cuantizacion_calidad.md merging/logs/tiempos_cuantizacion.json merging/logs/cuant_*.log
from google.colab import files
files.download("resultados_cuantizacion.zip")